# 🗂️ Notebook 1 — Organisation & Traitement des datasets

## Objectifs
1. **Inventaire** complet de tous les datasets disponibles
2. **Réorganisation par catégorie** : 15 dossiers structurés
3. **Traitement** des nouveaux fichiers (IRENA, CMM, Ember, SE4ALL, GLW...)
4. **Production** de fichiers cleaned standardisés (ISO, Année)

## 15 catégories
| Catégorie | Description |
|---|---|
| `atmosphere` | Climat dynamique, émissions CO2/CH4/N2O, qualité air |
| `hydrologie` | Eau (stress, accès, retraits), niveau mer, glace |
| `sol_ecologie` | Sol, biodiversité (IUCN, EPI), forêt, feux |
| `agriculture` | Cultures végétales, intrants, suitability, SPAM |
| `elevage` | Animaux, viande, lait, œufs, maladies WAHIS |
| `peche` | Production halieutique, aquaculture (FAO) |
| `energie` | Production, conso, renouvelables, fossile, IRENA, Ember |
| `geologie` | Minéraux, séismes, volcans |
| `demographie` | Population, santé, vie, fécondité, mortalité |
| `economie` | PIB, commerce, finances, valeur production |
| `climat_indices` | ENSO, NAO, AMO, PDO, indices NOAA |
| `worldclim` | WorldClim BIO 1-19, T, P, élévation |
| `geographie` | Coords pays, centroides, features physiques |
| `shared` | datasets finaux, country_clusters, EcoCrop |
| `misc` | Utilitaires divers |

## 1. Imports & configuration

In [ ]:
import os, sys, io, shutil, glob, zipfile
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW = 'data/raw'
CLN = 'data/cleaned'
CATEGORIES = ['atmosphere','hydrologie','sol_ecologie','agriculture','elevage',
              'peche','energie','geologie','demographie','economie',
              'climat_indices','worldclim','geographie','shared','misc']

## 2. Inventaire actuel

In [ ]:
for base in [RAW, CLN]:
    print(f'\n=== {base} ===')
    for c in CATEGORIES:
        d = os.path.join(base, c)
        if os.path.isdir(d):
            n_csv = len([f for f in os.listdir(d) if f.endswith('.csv')])
            n_other = len(os.listdir(d)) - n_csv
            print(f'  {c:18s} : {n_csv:3d} csv + {n_other:2d} autres')

## 3. Fonction de catégorisation

Classifie chaque fichier dans une des 15 catégories selon des regex sur le nom de fichier.

In [ ]:
def categorize(filename):
    n = filename.lower()
    if any(k in n for k in ['edgar_','ch4','co2','n2o','f-gas','emissions','co2_','methane','ozone',
                              'climate_watch','temperature','pm25','mean_temperature','variation_temp',
                              'global_co2','berkeley_earth','owid_co','owid_methane','owid_n2o',
                              'owid_avg_monthly_temp','owid_temp_anomaly','worldclim_bio_extra',
                              'download_cmm','cmm_']):
        return 'atmosphere'
    if any(k in n for k in ['aqueduct','aquastat','hydro','water','sea_level','ocean','psmsl',
                              'cobe','grace','wri_','groundwater','tide','wb_freshwater',
                              'wb_safe_water','wb_sanitation','owid_water']):
        return 'hydrologie'
    if any(k in n for k in ['forest','tree_cover','ecologie','biodiv','iucn','epi_','firms',
                              'fire','ndvi','modis','viirs','gbif','biomass','pedologie','sols',
                              'soil','owid_forest','wb_forest','bilan_nutritif','wood_density',
                              'deforest']):
        return 'sol_ecologie'
    if any(k in n for k in ['spam','crop','yield','fertili','pesticid','land_use_input','landuse',
                              'intrants','arable','agricultur','cereal','vegetable','fruit','plant',
                              'sugar','cotton','tobacco','owid_agri','owid_cereal','owid_food',
                              'wb_agri','wb_cereal','wb_arable','wb_food_production','wb_fertili',
                              'wb_irrigat','production_cultures','ecocrop','gaez','crop_calendar']):
        return 'agriculture'
    if any(k in n for k in ['livestock','meat','milk','egg','cattle','poultry','sheep','pig',
                              'glw','production_animaux','owid_meat','owid_milk','owid_egg',
                              'wahis','donnees quantitatives']):
        return 'elevage'
    if any(k in n for k in ['fish','aqua_culture','aquaculture','fishery','peche','marine_protected',
                              'fishstat','globalproductionfish']):
        return 'peche'
    if any(k in n for k in ['irena','ember','electric','energy','power','grid','solar','wind',
                              'hydroelec','coal','oil','gas_prod','fossil','renewable','reshare',
                              'heatgen','r-elec','r_elec','monthly_capacity','yearly_full_release',
                              'se4all','sustainable_energy','targets_download','wb_electric',
                              'wb_energy','wb_renewable','wb_coal_rents','wb_oil_rents',
                              'wb_natgas_rents','wb_mineral_rents','wb_forest_rents','ember_yearly',
                              'global_power_plants']):
        return 'energie'
    if any(k in n for k in ['mrds','mineral','earthquake','seismic','volcano','volcan',
                              'tectonic','geolog','iron_mine','oil_gas','coal_mine']):
        return 'geologie'
    if any(k in n for k in ['population','mortality','fertility','birth','death','child_mort',
                              'life_exp','migrant','hunger','stunting','wasting','owid_birth',
                              'owid_death','owid_pop','wb_population','wb_birth','wb_death',
                              'wb_child_mortality','wb_life_exp','wb_dependency','wb_pop_',
                              'wb_stunting','wb_wasting','wb_overweight','wb_adult','wb_infant',
                              'wb_school','wb_employ_','wb_unemployment','wb_urban',
                              'wb_internet','wb_mobile','wb_broadband','wb_hospital','wb_physic',
                              'wb_health','wb_hiv','wb_malaria','wb_deaths_communic',
                              'global_hunger','who_pm25','who_ambient']):
        return 'demographie'
    if any(k in n for k in ['gdp','econom','trade','commerce','finance','debt','inflation',
                              'gini','poverty','value_of_production','value_production','wb_gdp',
                              'wb_trade','wb_inflation','wb_gini','wb_poverty','wb_public_debt',
                              'wb_rd_expenditure','wb_manuf','wb_services','wb_agri_value',
                              'fao_value_of','fao_value_production']):
        return 'economie'
    if any(k in n for k in ['climate_index','enso','nao','amo','pdo','soi','oni','ao_lag','ao_index']):
        return 'climat_indices'
    if 'worldclim' in n or n.startswith(('bio','prec','tmax','tmin','tvag','wind_','solrad','vapr','elev')):
        return 'worldclim'
    if n.startswith(('dataset_final','country_','ecocrop','datset_final')):
        return 'shared'
    if any(k in n for k in ['topographie','relief','elevation','slope','centroid','physical_features']):
        return 'geographie'
    return 'misc'

# Test
for f in ['edgar_2025_ch4.csv','wb_population_total.csv','spam2020_v2_yield_by_country.csv',
          'iucn_species_by_country.csv','irena_renewable_targets.csv']:
    print(f'{f:50s} → {categorize(f)}')

## 4. Réorganisation (déplace fichiers)

In [ ]:
# Le script organize_and_process.py a déjà réorganisé.
# Pour ré-exécuter manuellement :
print('Réorganisation déjà appliquée. Pour relancer :')
print('  !python organize_and_process.py')

## 5. Traitement des nouveaux fichiers

### IRENA — Renewable Targets
Format Excel multi-sheets, parse les cibles renouvelables par pays.

In [ ]:
src = f'{RAW}/energie/targets_download.xlsx'
if os.path.exists(src):
    df = pd.read_excel(src, sheet_name='raw_data_long', engine='openpyxl')
    print(f'IRENA targets : {df.shape}')
    print(df.head(3))
else:
    print(f'⚠️ {src} introuvable')

### CMM — Coal Mine Methane Emissions

In [ ]:
src = f'{RAW}/atmosphere/download_cmm_emissions.xlsx'
if os.path.exists(src):
    df = pd.read_excel(src, sheet_name='emissions', engine='openpyxl')
    print(f'CMM emissions : {df.shape}')
    print(df.head(3))
else:
    print('⚠️ pas dispo')

### Ember Yearly Electricity

In [ ]:
src = f'{RAW}/energie/yearly_full_release_long_format.csv'
if os.path.exists(src):
    df = pd.read_csv(src, low_memory=False)
    print(f'Ember : {df.shape}')
    print(f'Catégories : {df["Category"].value_counts().head().to_dict()}')
else:
    print('⚠️ pas dispo')

### SE4ALL — Sustainable Energy for All (WB)

In [ ]:
src = f'{RAW}/energie/P_Data_Extract_From_Sustainable_Energy_for_All.zip'
if os.path.exists(src):
    with zipfile.ZipFile(src) as z:
        print(f'Fichiers : {z.namelist()}')
        with z.open(z.namelist()[0]) as f:
            df = pd.read_csv(f, low_memory=False)
            print(f'Shape : {df.shape}')
            indicators = [c for c in df.columns if '[' in c and ']' in c][:6]
            for ind in indicators:
                print(f'  - {ind[:80]}')

### WAHIS — Animal Disease Outbreaks

In [ ]:
for src in [f'{RAW}/elevage/Données quantitatives 2026-06-24.csv',
             f'{RAW}/demographie/Données quantitatives 2026-06-24.csv',
             f'{RAW}/Données quantitatives 2026-06-24.csv']:
    if os.path.exists(src):
        try: df = pd.read_csv(src, low_memory=False)
        except: df = pd.read_csv(src, low_memory=False, encoding='latin-1')
        print(f'WAHIS : {df.shape}')
        print(f'Maladies top 5 : {df["Maladie"].value_counts().head().to_dict()}')
        break
else:
    print('⚠️ WAHIS non trouvé')

## 6. Inventaire final des cleaned

In [ ]:
print('Fichiers cleaned par catégorie :\n')
total = 0
for c in CATEGORIES:
    d = os.path.join(CLN, c)
    if os.path.isdir(d):
        files = [f for f in os.listdir(d) if f.endswith('.csv')]
        total += len(files)
        if files:
            print(f'\n━━ {c} ({len(files)} csv) ━━')
            for f in sorted(files)[:8]:
                sz = os.path.getsize(os.path.join(d,f)) / 1024
                print(f'  {f:50s} ({sz:6.0f} KB)')
            if len(files) > 8:
                print(f'  ... et {len(files)-8} autres')
print(f'\nTOTAL : {total} fichiers cleaned')

## 7. Récapitulatif du pipeline

```
DATA SOURCES (data/raw/)
   │
   ├── atmosphere/        (EDGAR, CMM, NOAA, WorldClim temp)
   ├── hydrologie/        (Aqueduct, AQUASTAT, PSMSL)
   ├── sol_ecologie/      (EPI, IUCN, GBIF, FIRMS)
   ├── agriculture/       (SPAM, FAO Fertilizers, Pesticides, FAOSTAT)
   ├── elevage/           (FAO Animal Prod, GLW4, WAHIS)
   ├── peche/             (FAO Fish Production)
   ├── energie/           (IRENA, Ember, SE4ALL)
   ├── geologie/          (USGS MRDS, séismes, volcans)
   ├── demographie/       (WB, OWID, WHO)
   ├── economie/          (FAO Value, WB GDP)
   ├── climat_indices/    (NOAA ENSO, NAO, AMO, PDO)
   ├── worldclim/         (WorldClim BIO 1-19)
   ├── geographie/        (Centroids, slopes)
   └── shared/            (datasets finaux, country_clusters, EcoCrop)

   ↓ TRAITEMENT (scripts Python)

CLEANED DATA (data/cleaned/) — même structure
   ↓
   datasets finaux multi-couches (dataset_final_v*_couche*.csv)
```

→ Voir Notebook 2 (exploration) pour visualisations + analyses